# Analysis of the Australian Frogs dataset

Group: A Leap of Faith

Members: 
Vanshika Dhamija (A0333989H), Aparna Krishna(A0326884N), Apurwa Kumari (A0333260R), Aditya Vijindra Pandhari (A0328873R), Shreya Sriram(A0327236E)

### TOC
TBD

### 1. Introduction

### 2. Dataset

In [ ]:
import pandas as pd
import os

DATA_DIR = 'data'
FROGID_PATH = os.path.join(DATA_DIR, 'frogID_data.csv')
FROG_NAMES_PATH = os.path.join(DATA_DIR, 'frog_names.csv')

FROGID_URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-02/frogID_data.csv'
FROG_NAMES_URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-02/frog_names.csv'

# Create data directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# Load from local cache if available, otherwise download and save
if os.path.exists(FROGID_PATH) and os.path.exists(FROG_NAMES_PATH):
    print("Loading data from local cache...")
    frogID_data = pd.read_csv(FROGID_PATH)
    frog_names = pd.read_csv(FROG_NAMES_PATH)
else:
    print("Local cache not found. Downloading from TidyTuesday")
    frogID_data = pd.read_csv(FROGID_URL)
    frog_names = pd.read_csv(FROG_NAMES_URL)
    frogID_data.to_csv(FROGID_PATH, index=False)
    frog_names.to_csv(FROG_NAMES_PATH, index=False)
    print(f"Data saved to {DATA_DIR}/")

print(f"frogID_data shape: {frogID_data.shape}")
print(f"frog_names shape: {frog_names.shape}")

Local cache not found. Downloading from TidyTuesday


In [ ]:
# check the number of rows and columns in the datasets
print(f"frogID_data shape: {frogID_data.shape}")
print(f"frog_names shape: {frog_names.shape}")

In [ ]:
# check the top 5 rows of the frogID_data dataset
print("Top 5 rows of frogID_data:")
print(frogID_data.head())

# check the top 5 rows of the frog_names dataset
print("Top 5 rows of frog_names:")
print(frog_names.head())

In [ ]:
# Print the column names of both datasets
print("Columns in frogID_data:")
print(frogID_data.columns)

print("Columns in frog_names:")
print(frog_names.columns)

### 3. Hypothesis

Given that the dataset comprises of frog names, geographical coordinates, dates they were found on, the data is rich both spatially and seasonally. This leads us to perform a thorough data analysis to answer the question: "Do frog observations in Australia shift geographically across seasons, with higher activity in northern regions during the dry season and southern regions during the wetter, cooler months?"

### 4. Preprocessing

#### 4.1. Checking the overlap in scientific names between the frog ID dataset and frog names dataset 

In [ ]:
# check if there are any scientific names in the frogID_data dataset that do not have a match in the frog_names dataset
missing_names = frogID_data[~frogID_data['scientificName'].isin(frog_names['scientificName'])]
print(f"Number of scientific names in frogID_data without a match in frog_names: {missing_names.shape[0]}")

# Get the unique missing names values
unique_missing_names = missing_names['scientificName'].unique()
print("Unique scientific names in frogID_data without a match in frog_names:")
print(unique_missing_names)

In [ ]:
# Which of the unique missing names present in the merged dataset have more than 1 row
missing_names_counts = missing_names['scientificName'].value_counts()
duplicates = missing_names_counts[missing_names_counts > 1]
print("Unique missing names present in the merged dataset with more than 1 row:")
print(duplicates)

#### 4.2. Exploring possible matches for the missing scientific names and cleaning unmatchable data

In [ ]:
# Check for whitespace differences
for name in unique_missing_names:
    print(repr(name))

# Also check frog_names for similar entries
for name in unique_missing_names:
    partial = name.split()[0]  # just genus
    matches = frog_names[frog_names['scientificName'].str.startswith(partial)]
    print(f"\n{name} → possible matches:")
    print(matches['scientificName'].values)

The above distribution indicates that there are possible matches for 5 of the 6 scientific names. Since Lechriodus fletcheri doesn't have a match that can be determined lexicographically and only has 19 rows (~0.014% of the data in frog IDs), dropping the data.

In [ ]:
frogID_data_clean = frogID_data[frogID_data['scientificName'] != 'Lechriodus fletcheri']

Performing a surgical fix for the other missing frogIDs based on the pattern search above

In [ ]:
# Fix 1: Direct name corrections (typos/gender variants)
name_corrections = {
    'Philoria sphagnicola': 'Philoria sphagnicolus',
    'Cyclorana platycephala': 'Cyclorana platycephalus',
}
frogID_data['scientificName'] = frogID_data['scientificName'].replace(name_corrections)

# Fix 2: Subspecies → species-level rollup
# Map the subspecies entries in frog_names back to species level for joining
subspecies_rollup = {
    'Heleioporus australiacus australiacus': 'Heleioporus australiacus',
    'Heleioporus australiacus flavopunctatus': 'Heleioporus australiacus',
    'Litoria verreauxii alpina': 'Litoria verreauxii',
    'Litoria verreauxii verreauxii': 'Litoria verreauxii',
    'Limnodynastes dumerilii dumerilii': 'Limnodynastes dumerilii',
    'Limnodynastes dumerilii fryi': 'Limnodynastes dumerilii',
    'Limnodynastes dumerilii insularis': 'Limnodynastes dumerilii',
    'Limnodynastes dumerilii variegatus': 'Limnodynastes dumerilii',
}

# Create a cleaned frog_names with rolled-up species names
frog_names_clean = frog_names.copy()
frog_names_clean['scientificName'] = frog_names_clean['scientificName'].replace(subspecies_rollup)

# Where multiple subspecies map to the same species, keep the first entry
frog_names_clean = frog_names_clean.drop_duplicates(subset='scientificName', keep='first')

#### 4.3. Ensuring occurenceIDs in the frogIDs dataset are unique

In [ ]:
print(frogID_data_clean['occurrenceID'].duplicated().sum())

#### 4.4. Check coordinate validity

In [ ]:
invalid_coords = frogID_data_clean[
    (frogID_data_clean['decimalLatitude'] < -44) | (frogID_data_clean['decimalLatitude'] > -10) |
    (frogID_data_clean['decimalLongitude'] < 113) | (frogID_data_clean['decimalLongitude'] > 154)
]
print(f"Rows with suspicious coordinates: {invalid_coords.shape[0]}")

#### 4.7. Check state province for null/invalid values

In [ ]:
frogID_data_clean['stateProvince'].value_counts(dropna=False)

#### 4.6. Parse eventDate to perform seasonal analysis

In [ ]:
frogID_data_clean['eventDate'] = pd.to_datetime(frogID_data_clean['eventDate'])
frogID_data_clean['month'] = frogID_data_clean['eventDate'].dt.month
frogID_data_clean['month_name'] = frogID_data_clean['eventDate'].dt.strftime('%b')

#### 4.7. Checking the distribution of coordinate uncertainity

In [ ]:
frogID_data_clean['coordinateUncertaintyInMeters'].describe()

#### 4.8. Merging the cleaned datasets

In [ ]:
frog_data_merged = pd.merge(frogID_data, frog_names_clean, on='scientificName', how='left')

In [ ]:
# Ensuring that no columns have null data

### 5. Plotting

Plot 1: Bar chart: Observation counts by month

This sets the scene — showing when frog activity peaks across the year before you get into where. It answers the "higher activity" part of the hypothesis and gives the reader the temporal context needed to interpret the spatial plots that follow. A bar chart is ideal here because months are discrete categories and you want the reader to compare heights directly.

Plot 2: Line chart: Observation counts by month, broken down by region

Variables: month_name, count of occurrenceID, stateProvince (or a derived region column grouping states into North/South/East/West)
This is the analytical core of your hypothesis. By splitting the monthly trend by region, the reader can directly see whether northern and southern activity peaks at different times of year. A line chart is ideal because it emphasises the trend shape per region and lets multiple lines sit cleanly on one chart without overcrowding.
A tip — stateProvince has too many categories to plot cleanly as-is. Consider grouping into broader regions:
```
pythonregion_map = {
    'Queensland': 'North',
    'Northern Territory': 'North',
    'Western Australia': 'North/West',
    'New South Wales': 'South/East',
    'Victoria': 'South/East',
    'South Australia': 'South',
    'Tasmania': 'South',
    'Australian Capital Territory': 'South/East'
}
frog_data_merged['region'] = frog_data_merged['stateProvince'].map(region_map)
```


Plot 3 — Choropleth or hexbin map: Observation density by region and season
**Variables:** `decimalLatitude`, `decimalLongitude`, `month` (grouped into seasons)

This is the spatial payoff — the previous two plots told the story numerically, this one shows it geographically. Facet the map into 2 or 3 seasonal panels (e.g. Summer/Autumn/Winter/Spring, or simply Wet vs Dry) so the reader can visually see the density shift between north and south across seasons.

---

## How the three plots work together
```
Plot 1 (Bar)         Plot 2 (Line)            Plot 3 (Map)
"When are frogs  →   "Which regions drive  →  "Show me where
most active?"        those seasonal peaks?"    this looks like
                                               on a map"